# Validate Emotion Vectors


In [ ]:
import json
import torch
import numpy as np
import torch.nn.functional as F
from pathlib import Path
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM
import sys
sys.path.append("..")
from config import DEFAULT_MODEL, TARGET_LAYER, TOKEN_START, BATCH_SIZE, VECTORS_OUT, STORIES_JSONL

N_HOLDOUT = 40  # stories per emotion to use for validation
print(f"Config loaded. Holding out last {N_HOLDOUT} stories per emotion.")


## Load Held-Out Stories


In [ ]:
# Group all stories by emotion
all_stories = defaultdict(list)
with open(STORIES_JSONL) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        all_stories[rec["emotion"]].append(rec["text"])

# Hold out the last N_HOLDOUT per emotion
eval_pairs = []  # (text, emotion)
for emotion, texts in sorted(all_stories.items()):
    holdout = texts[-N_HOLDOUT:]
    for t in holdout:
        eval_pairs.append((t, emotion))
    print(f"  {emotion:<20} {len(texts):>4} total  →  {len(holdout)} held out")

print(f"\nTotal validation stories: {len(eval_pairs)}")


## Load Emotion Vectors


In [ ]:
vecs_dict = torch.load(VECTORS_OUT, map_location="cpu", weights_only=False)
emotions  = sorted(vecs_dict.keys())
mat       = torch.stack([vecs_dict[e] for e in emotions], dim=0).float()
mat       = F.normalize(mat, dim=1)
emotion_to_idx = {e: i for i, e in enumerate(emotions)}
print(f"{len(emotions)} emotion vectors loaded, dim={mat.shape[1]}")


## Load Model


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    DEFAULT_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("Model loaded.")


## Encode and Project


In [ ]:
def mean_pool_from(hidden, start):
    sliced = hidden[start:]
    if sliced.shape[0] == 0:
        sliced = hidden
    return sliced.mean(dim=0)

@torch.inference_mode()
def encode_texts(texts, layer, token_start, batch_size):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=512).to(device)
        out = model(**enc, output_hidden_states=True, use_cache=False)
        layer_h = out.hidden_states[layer].float().cpu()
        mask    = enc["attention_mask"].cpu()
        for b in range(layer_h.shape[0]):
            seq_len = mask[b].sum().item()
            h   = layer_h[b, :seq_len, :]
            vec = mean_pool_from(h, token_start)
            all_vecs.append(vec)
    return torch.stack(all_vecs, dim=0)

texts  = [p[0] for p in eval_pairs]
labels = [p[1] for p in eval_pairs]

print(f"Encoding {len(texts)} stories...")
activations = encode_texts(texts, layer=TARGET_LAYER, token_start=TOKEN_START, batch_size=BATCH_SIZE)
activations = F.normalize(activations.float(), dim=1)
scores = activations @ mat.T  # (N, E)
print(f"Done. Scores shape: {scores.shape}")


## Results


In [ ]:
def rank_of_target(score_row, target_idx):
    return score_row.argsort(descending=True).tolist().index(target_idx) + 1

results = {}
all_ranks = []

for emotion in emotions:
    target_idx = emotion_to_idx[emotion]
    idxs  = [i for i, l in enumerate(labels) if l == emotion]
    ranks = [rank_of_target(scores[i], target_idx) for i in idxs]
    all_ranks.extend(ranks)
    results[emotion] = {
        "n":           len(idxs),
        "top1_acc":    sum(r == 1 for r in ranks) / len(ranks),
        "top3_acc":    sum(r <= 3 for r in ranks) / len(ranks),
        "top5_acc":    sum(r <= 5 for r in ranks) / len(ranks),
        "mean_rank":   float(np.mean(ranks)),
        "mean_cosine": float(scores[idxs, target_idx].mean()),
    }

random_baseline = 1 / len(emotions)
overall_top1 = sum(r == 1 for r in all_ranks) / len(all_ranks)
overall_top3 = sum(r <= 3 for r in all_ranks) / len(all_ranks)
overall_top5 = sum(r <= 5 for r in all_ranks) / len(all_ranks)

print(f"Overall top-1 : {overall_top1:.1%}  (random baseline: {random_baseline:.1%})")
print(f"Overall top-3 : {overall_top3:.1%}")
print(f"Overall top-5 : {overall_top5:.1%}")
print(f"Mean rank     : {np.mean(all_ranks):.1f} / {len(emotions)}")
print()
print(f"  {'Emotion':<20} {'N':>4}  {'Top1':>6}  {'Top3':>6}  {'Top5':>6}  {'MeanRank':>9}  {'CosSim':>7}")
print("  " + "-" * 65)
for emotion, r in sorted(results.items(), key=lambda x: -x[1]['top1_acc']):
    print(f"  {emotion:<20} {r['n']:>4}  {r['top1_acc']:>6.1%}  {r['top3_acc']:>6.1%}  "
          f"{r['top5_acc']:>6.1%}  {r['mean_rank']:>9.1f}  {r['mean_cosine']:>7.4f}")


## Confusion Matrix


In [ ]:
# For each emotion, show the top 3 most common predictions
print(f"{'Emotion':<20}  Most common predictions (when wrong)")
print("-" * 70)
for emotion in sorted(emotions):
    target_idx = emotion_to_idx[emotion]
    idxs = [i for i, l in enumerate(labels) if l == emotion]
    wrong = []
    for i in idxs:
        pred_idx = scores[i].argmax().item()
        if pred_idx != target_idx:
            wrong.append(emotions[pred_idx])
    if wrong:
        from collections import Counter
        top_wrong = Counter(wrong).most_common(3)
        top_str = ", ".join(f"{e} ({n})" for e, n in top_wrong)
    else:
        top_str = "none — 100% correct!"
    top1 = results[emotion]['top1_acc']
    print(f"  {emotion:<20} {top1:>5.1%} top1  →  {top_str}")


## Save Results


In [ ]:
out_dir = Path("data")
out_dir.mkdir(exist_ok=True)

import json
overall = {
    "top1_acc": overall_top1,
    "top3_acc": overall_top3,
    "top5_acc": overall_top5,
    "mean_rank": float(np.mean(all_ranks)),
    "n_emotions": len(emotions),
    "n_examples": len(eval_pairs),
    "random_top1_baseline": random_baseline,
}
with open(out_dir / "validation_results.json", "w") as f:
    json.dump({"overall": overall, "per_emotion": results}, f, indent=2)

print("Saved -> data/validation_results.json")
